In [ ]:
!pip install librosa matplotlib numpy pandas

import zipfile
import os
import glob

import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np


# 1. Unzip the uploaded dataset
zip_file = "dataset.zip"
extract_path = "./dataset_files"

print("Extracting dataset...")

with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Unzipping complete!")


# 2. Check for nested zip files
nested_zips = glob.glob(
    f"{extract_path}/**/*.zip",
    recursive=True
)

for n_zip in nested_zips:
    with zipfile.ZipFile(n_zip, 'r') as zip_ref:
        zip_ref.extractall(os.path.dirname(n_zip))


# 3. Find all WAV files
all_wavs = glob.glob(
    f"{extract_path}/**/*.wav",
    recursive=True
)

print(f"\nTotal WAV Audio Files Found: {len(all_wavs)}")

print("\nSample Audio Files:")
for wav in all_wavs[:5]:
    print(os.path.basename(wav))

Extracting dataset...
Unzipping complete!

Total WAV Audio Files Found: 5477

Sample Audio Files:
id_161_sound_27.wav
id_128_sound_31.wav
id_161_sound_12.wav
id_156_sound_1.wav
id_163_sound_27.wav


In [ ]:
# Categorize Tomato samples

tomato_dry = [
    f for f in all_wavs
    if "tomato" in f.lower()
    and "dry" in f.lower()
]

tomato_cut = [
    f for f in all_wavs
    if "tomato" in f.lower()
    and "cut" in f.lower()
]

tomato_control = [
    f for f in all_wavs
    if "tomato" in f.lower()
    and (
        "control" in f.lower()
        or "baseline" in f.lower()
    )
]


print("--- Tomato Dataset Breakdown ---")

print(
    f"• Dry Tomato (Drought Stress): "
    f"{len(tomato_dry)} files"
)

print(
    f"• Cut Tomato (Physical Stress): "
    f"{len(tomato_cut)} files"
)

print(
    f"• Control Tomato: "
    f"{len(tomato_control)} files"
)


# Check unique subfolders
dirs = set([
    os.path.dirname(f)
    for f in all_wavs
    if "tomato" in f.lower()
])

print("\nTomato Directories Found:")

for d in sorted(dirs):
    print(d)

--- Tomato Dataset Breakdown ---
• Dry Tomato (Drought Stress): 1622 files
• Cut Tomato (Physical Stress): 660 files
• Control Tomato: 0 files

Tomato Directories Found:
./dataset_files/Tomato Cut
./dataset_files/Tomato Dry


In [ ]:
# ============================================================
# AUDIO AUGMENTATION FUNCTIONS
# ============================================================

# 1. Add Noise
def add_noise(y, noise_factor=0.01):

    noise = np.random.randn(len(y))

    return y + noise_factor * noise


# 2. Time Shift
def time_shift(y, shift_max=0.1):

    shift = int(
        np.random.uniform(
            -shift_max,
            shift_max
        ) * len(y)
    )

    return np.roll(y, shift)


# 3. Time Stretch
def time_stretch(y, rate):

    try:

        stretched = librosa.effects.time_stretch(
            y,
            rate=rate
        )

        # Keep original length

        if len(stretched) > len(y):

            stretched = stretched[:len(y)]

        else:

            stretched = np.pad(
                stretched,
                (0, len(y) - len(stretched))
            )

        return stretched

    except Exception:

        return y


# 4. SpecAugment
def apply_spec_augment(
    S,
    time_mask_size=10,
    frequency_mask_size=10
):

    S_aug = S.copy()

    # Frequency masking
    freq_width = np.random.randint(
        0,
        min(
            frequency_mask_size,
            S.shape[0]
        ) + 1
    )

    if freq_width > 0:

        freq_start = np.random.randint(
            0,
            max(
                1,
                S.shape[0] - freq_width
            )
        )

        S_aug[
            freq_start:freq_start + freq_width,
            :
        ] = S_aug.min()


    # Time masking
    time_width = np.random.randint(
        0,
        min(
            time_mask_size,
            S.shape[1]
        ) + 1
    )

    if time_width > 0:

        time_start = np.random.randint(
            0,
            max(
                1,
                S.shape[1] - time_width
            )
        )

        S_aug[
            :,
            time_start:time_start + time_width
        ] = S_aug.min()


    return S_aug


print("Augmentation functions are ready!")

Augmentation functions are ready!


In [ ]:
# ============================================================
# AUDIO → MEL SPECTROGRAM
# ============================================================

def save_spectrogram(
    audio_path,
    save_path,
    augmentation=None
):

    try:

        # Load audio
        y, sr = librosa.load(
            audio_path,
            sr=None
        )


        # -----------------------------
        # Audio Augmentation
        # -----------------------------

        if augmentation == "noise":

            y = add_noise(
                y,
                noise_factor=0.01
            )


        elif augmentation == "shift":

            y = time_shift(
                y,
                shift_max=0.1
            )


        elif augmentation == "stretch":

            y = time_stretch(
                y,
                rate=np.random.uniform(
                    0.95,
                    1.05
                )
            )


        # -----------------------------
        # Mel Spectrogram
        # -----------------------------

        S = librosa.feature.melspectrogram(
            y=y,
            sr=sr,
            n_mels=128
        )

        S_dB = librosa.power_to_db(
            S,
            ref=np.max
        )


        # -----------------------------
        # SpecAugment
        # -----------------------------

        if augmentation == "specaugment":

            S_dB = apply_spec_augment(
                S_dB,
                time_mask_size=10,
                frequency_mask_size=10
            )


        # -----------------------------
        # Save Image
        # -----------------------------

        plt.figure(
            figsize=(3, 3)
        )

        plt.axis("off")

        librosa.display.specshow(
            S_dB,
            sr=sr
        )

        plt.savefig(
            save_path,
            bbox_inches="tight",
            pad_inches=0
        )

        plt.close()


    except Exception as e:

        print(
            f"Error processing {audio_path}: {e}"
        )


print("Spectrogram function is ready!")

Spectrogram function is ready!


In [ ]:
# ============================================================
# CREATE OUTPUT FOLDERS
# ============================================================

output_path = "./processed_dataset_augmented"

os.makedirs(
    f"{output_path}/Dry",
    exist_ok=True
)

os.makedirs(
    f"{output_path}/Cut",
    exist_ok=True
)


# ============================================================
# PROCESS DRY TOMATO
# ============================================================

print(
    f"Processing Dry Tomato "
    f"({len(tomato_dry)} files)..."
)

for i, path in enumerate(tomato_dry):

    # Original
    save_spectrogram(
        path,
        f"{output_path}/Dry/"
        f"dry_{i}_original.png"
    )

    # Noise
    save_spectrogram(
        path,
        f"{output_path}/Dry/"
        f"dry_{i}_noise.png",
        augmentation="noise"
    )

    # Time Shift
    save_spectrogram(
        path,
        f"{output_path}/Dry/"
        f"dry_{i}_shift.png",
        augmentation="shift"
    )

    # Time Stretch
    save_spectrogram(
        path,
        f"{output_path}/Dry/"
        f"dry_{i}_stretch.png",
        augmentation="stretch"
    )

    # SpecAugment
    save_spectrogram(
        path,
        f"{output_path}/Dry/"
        f"dry_{i}_specaugment.png",
        augmentation="specaugment"
    )

    if (i + 1) % 100 == 0:

        print(
            f"Dry: "
            f"{i + 1}/{len(tomato_dry)}"
        )


# ============================================================
# PROCESS CUT TOMATO
# ============================================================

print(
    f"\nProcessing Cut Tomato "
    f"({len(tomato_cut)} files)..."
)

for i, path in enumerate(tomato_cut):

    # Original
    save_spectrogram(
        path,
        f"{output_path}/Cut/"
        f"cut_{i}_original.png"
    )

    # Noise
    save_spectrogram(
        path,
        f"{output_path}/Cut/"
        f"cut_{i}_noise.png",
        augmentation="noise"
    )

    # Time Shift
    save_spectrogram(
        path,
        f"{output_path}/Cut/"
        f"cut_{i}_shift.png",
        augmentation="shift"
    )

    # Time Stretch
    save_spectrogram(
        path,
        f"{output_path}/Cut/"
        f"cut_{i}_stretch.png",
        augmentation="stretch"
    )

    # SpecAugment
    save_spectrogram(
        path,
        f"{output_path}/Cut/"
        f"cut_{i}_specaugment.png",
        augmentation="specaugment"
    )

    if (i + 1) % 100 == 0:

        print(
            f"Cut: "
            f"{i + 1}/{len(tomato_cut)}"
        )


# ============================================================
# FINAL RESULTS
# ============================================================

dry_images = glob.glob(
    f"{output_path}/Dry/*.png"
)

cut_images = glob.glob(
    f"{output_path}/Cut/*.png"
)


print("\n====================================")
print("AUGMENTATION COMPLETED!")
print("====================================")

print(
    f"Dry Spectrograms: "
    f"{len(dry_images)}"
)

print(
    f"Cut Spectrograms: "
    f"{len(cut_images)}"
)

print("\nSaved in:")
print(output_path)

Processing Dry Tomato (1622 files)...


/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1001
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/librosa/feature/spectral.py:2148: UserWarning: Empty filters detected in mel frequency basis. Some channels will produce empty responses. Try increasing your sampling rate (and fmax) or reducing n_mels.
  mel_basis = filters.mel(sr=sr, n_fft=n_fft, **kwargs)
/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1001
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1001
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/librosa/feature/spectral.py:2148: UserWarning: Empty filters detected in mel frequency basis. Some channels will produce empty responses. Try increasing your sampling rate (and fmax) or reducin

Dry: 100/1622
Dry: 200/1622
Dry: 300/1622
Dry: 400/1622
Dry: 500/1622
Dry: 600/1622
Dry: 700/1622
Dry: 800/1622
Dry: 900/1622
Dry: 1000/1622
Dry: 1100/1622
Dry: 1200/1622
Dry: 1300/1622
Dry: 1400/1622
Dry: 1500/1622
Dry: 1600/1622

Processing Cut Tomato (660 files)...
Cut: 100/660
Cut: 200/660
Cut: 300/660
Cut: 400/660
Cut: 500/660
Cut: 600/660

AUGMENTATION COMPLETED!
Dry Spectrograms: 8110
Cut Spectrograms: 3300

Saved in:
./processed_dataset_augmented


In [ ]:
import os
import shutil
import random

# Paths
source_path = "./processed_dataset_augmented"
balanced_path = "./balanced_dataset"

dry_path = os.path.join(source_path, "Dry")
cut_path = os.path.join(source_path, "Cut")

balanced_dry = os.path.join(balanced_path, "Dry")
balanced_cut = os.path.join(balanced_path, "Cut")

os.makedirs(balanced_dry, exist_ok=True)
os.makedirs(balanced_cut, exist_ok=True)

# Get all images
dry_images = [
    os.path.join(dry_path, f)
    for f in os.listdir(dry_path)
    if f.endswith(".png")
]

cut_images = [
    os.path.join(cut_path, f)
    for f in os.listdir(cut_path)
    if f.endswith(".png")
]

# Use the same number from both classes
random.seed(42)

number_per_class = min(len(dry_images), len(cut_images))

selected_dry = random.sample(dry_images, number_per_class)
selected_cut = random.sample(cut_images, number_per_class)

# Copy images
for img in selected_dry:
    shutil.copy(img, balanced_dry)

for img in selected_cut:
    shutil.copy(img, balanced_cut)

print("BALANCING COMPLETED!")
print("--------------------")
print("Dry images:", len(os.listdir(balanced_dry)))
print("Cut images:", len(os.listdir(balanced_cut)))
print("Total images:",
      len(os.listdir(balanced_dry)) + len(os.listdir(balanced_cut)))

print("\nSaved in:")
print(balanced_path)

BALANCING COMPLETED!
--------------------
Dry images: 3300
Cut images: 3300
Total images: 6600

Saved in:
./balanced_dataset


In [ ]:
from google.colab import drive
import shutil
import os

# 1. Connect Google Drive
drive.mount('/content/drive')

# 2. Create one folder in Google Drive
drive_folder = '/content/drive/MyDrive/Plant_Whisper_AI'
os.makedirs(drive_folder, exist_ok=True)

# 3. Files/folders to upload
items = [
    './dataset.zip',
    './balanced_dataset',
    './processed_dataset_augmented'
]

# 4. Copy everything to Google Drive
for item in items:
    if os.path.exists(item):
        destination = os.path.join(
            drive_folder,
            os.path.basename(item)
        )

        if os.path.isdir(item):
            shutil.copytree(item, destination, dirs_exist_ok=True)
        else:
            shutil.copy2(item, destination)

        print("Uploaded:", os.path.basename(item))
    else:
        print("NOT FOUND:", item)

print("\n==============================")
print("ALL FILES UPLOADED SUCCESSFULLY!")
print("==============================")
print("Google Drive folder:")
print(drive_folder)

Mounted at /content/drive
Uploaded: dataset.zip
Uploaded: balanced_dataset
Uploaded: processed_dataset_augmented

ALL FILES UPLOADED SUCCESSFULLY!
Google Drive folder:
/content/drive/MyDrive/Plant_Whisper_AI


In [ ]:
import os
import shutil

# المسارات
source = "/content/processed_dataset_augmented/Dry"
destination = "/content/balanced_dataset/Dry"

# إنشاء مجلد Dry داخل balanced_dataset
os.makedirs(destination, exist_ok=True)

# نسخ كل ملفات Dry
for item in os.listdir(source):
    src = os.path.join(source, item)
    dst = os.path.join(destination, item)

    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        shutil.copy2(src, dst)

print("✅ Dry copied successfully!")
print("📁", destination)

✅ Dry copied successfully!
📁 /content/balanced_dataset/Dry
